In [1]:
import openai, json, requests

from dotenv import load_dotenv

load_dotenv()

client = openai.OpenAI()
messages = []
base_url = "https://nomad-movies-2.nomadcoders.workers.dev/"

In [2]:
def get_response_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.json()


def get_popular_movies():
    url = f"{base_url}movies"
    return get_response_data(url)


def get_movie_details(id):
    url = f"{base_url}movies/{id}"
    return get_response_data(url)


def get_movie_credits(id):
    url = f"{base_url}movies/{id}/credits"
    return get_response_data(url)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "A function to get popular movies.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "A function to get details of movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "A function to get credits of movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID.",
                    },
                },
                "required": ["id"],
            },
        },
    },
]

In [4]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=TOOLS
    )
    process_ai_response(response.choices[0].message)

In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Calling function: get_popular_movies with {}
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Peddi**
   - 개요: 1980년대 안드라 프라데시의 시골에서 열정적인 마을 주민이 스포츠를 통해 그의 공동체를 결합하고 강력한 라이벌에 맞서 그들의 자부심을 방어하려고 합니다.
   - 개봉일: 2026년 6월 3일
   - 평점: 6.5
   - ![포스터](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

2. **Obsession**
   - 개요: 심오한 사랑에 빠진 한 남자가 신비로운 "원 위시 윌로우"를 깨트려 그녀의 마음을 얻으려 할 때, 그는 자신이 원했던 것을 가지게 되지만, 그 소망이 어두운 대가를 치르게 한다는 것을 발견합니다.
   - 개봉일: 2026년 5월 13일
   - 평점: 7.9
   - ![포스터](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

3. **Hai Jawani Toh Ishq Hona Hai**
   - 개요: 결혼을 떠난 Jass는 충격적인 진실을 마주하게 되며 사랑과 충성심, 헌신의 진정한 의미를 직면하게 됩니다.
   - 개봉일: 2026년 6월 4일
   - 평점: 5.4
   - ![포스터](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

4. **The Unknown Man**
   - 개요: 루이, 플란더스 작가는 영감을 얻기 위해 코트 다쥐르에 자신을 고립시키려고 합니다.
   - 개봉일: 2021년 10월 16일
   - 평점: 8.2
   - ![포스터](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

5. **Mortal Komba